# 4. YOLOv8 - State-of-the-Art One-Stage Detection

## Su tien hoa tu YOLOv3 -> YOLOv8

### YOLOv4 (2020): CSPDarknet53, Mish activation, PANet, CIoU loss
### YOLOv5 (2020): PyTorch implementation, Focus layer, SPP, Auto-anchor
### YOLOv6 (2022): EfficientRep backbone, Rep-PAN neck, Task alignment
### YOLOv7 (2022): E-ELAN, compound model scaling
### YOLOv8 (2023, Ultralytics): **Anchor-free**, Decoupled Head, C2f module

## Cac cai tien chinh cua YOLOv8 so voi YOLOv3

| Feature | YOLOv3 | YOLOv8 |
|---------|--------|--------|
| **Backbone** | Darknet-53 | CSPDarknet (C2f modules) |
| **Neck** | FPN | FPN + PAN (bi-directional) |
| **Head** | Coupled (1 branch) | Decoupled (cls + reg rieng) |
| **Anchors** | Anchor-based (9 anchors) | **Anchor-free** |
| **Loss** | BCE + MSE | **CIoU + DFL + BCE** |
| **Label Assignment** | IoU-based | **Task-Aligned Assignment (TAL)** |

## Kien truc YOLOv8

```
Input Image (640x640)
        |
[CSPDarknet Backbone]
  - Conv layers voi C2f modules (Cross Stage Partial with 2 convolutions)
  - Bottleneck with residual connections
        |
   ┌────┴────┐
   |  [SPPF] |  (Spatial Pyramid Pooling Fast)
   └────┬────┘
        |
[FPN + PAN Neck]  (Feature Pyramid Network + Path Aggregation Network)
  - Top-down: truyen semantic features xuong
  - Bottom-up: truyen spatial features len
        |
   ┌────┼────┐
   |    |    |
 P3   P4   P5   (3 detection scales: 80x80, 40x40, 20x20)
   |    |    |
[Decoupled Head]  <-- Moi scale co head rieng
  - Classification branch (BCE loss)
  - Regression branch (CIoU + DFL loss)
  - KHONG co objectness branch (khac YOLOv3)
```

### Anchor-Free la gi?
- YOLOv3: Moi prediction duoc tinh tu 1 anchor box co san -> can chon anchor truoc
- YOLOv8: Predict truc tiep offset tu grid cell center -> khong can anchor
- Loi the: It hyperparameters, khong can k-means clustering, linh hoat hon

In [ ]:
import sys
sys.path.append("../src")

import os
import time
import json
import glob
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from tqdm import tqdm
from ultralytics import YOLO
from dataset import VOC_CLASSES

%matplotlib inline

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {DEVICE}")

DATA_ROOT = "../data"
RESULTS_DIR = "../results/yolov8"
os.makedirs(RESULTS_DIR, exist_ok=True)

# Duong dan den data.yaml (da tao o notebook 03)
data_yaml = os.path.join(DATA_ROOT, "VOC_YOLO", "data.yaml")
print(f"Dataset config: {data_yaml}")

## 4.1 Load va Fine-tune YOLOv8

Su dung YOLOv8m (medium) de co the so sanh cong bang voi Faster R-CNN va YOLOv3 ve so parameters.

In [ ]:
# Load YOLOv8m pretrained tren COCO
model = YOLO("yolov8m.pt")

print(f"Model: YOLOv8m")
print(f"Task: {model.task}")
n_params = sum(p.numel() for p in model.model.parameters())
print(f"Parameters: {n_params:,}")

# So sanh sizes
print(f"\nYOLOv8 Model Variants:")
print(f"  {'Variant':<10} {'Params':>12} {'mAP@0.5 (COCO)':>16}")
print(f"  {'yolov8n':<10} {'3.2M':>12} {'37.3':>16}")
print(f"  {'yolov8s':<10} {'11.2M':>12} {'44.9':>16}")
print(f"  {'yolov8m':<10} {'25.9M':>12} {'50.2':>16}  <- Chung ta dung")
print(f"  {'yolov8l':<10} {'43.7M':>12} {'52.9':>16}")
print(f"  {'yolov8x':<10} {'68.2M':>12} {'53.9':>16}")

In [ ]:
# Fine-tune YOLOv8m tren VOC
results = model.train(
    data=data_yaml,
    epochs=30,
    imgsz=640,
    batch=16,
    name="yolov8m_voc",
    project=RESULTS_DIR,
    pretrained=True,
    optimizer="SGD",
    lr0=0.01,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3,
    # Augmentation (giong YOLOv3 de so sanh cong bang)
    mosaic=1.0,
    flipud=0.0,
    fliplr=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    # Close mosaic augmentation o 10 epochs cuoi (YOLOv8 best practice)
    close_mosaic=10,
    # Save
    save=True,
    save_period=5,
    val=True,
    verbose=True,
)

## 4.2 Danh gia YOLOv8

In [ ]:
# Load best model
best_model_path = os.path.join(RESULTS_DIR, "yolov8m_voc", "weights", "best.pt")
best_model = YOLO(best_model_path)

# Evaluate
val_results = best_model.val(data=data_yaml, imgsz=640, batch=16, verbose=True)

print(f"\n{'='*60}")
print(f"  YOLOv8m - Evaluation Results")
print(f"{'='*60}")
print(f"  mAP@0.5:      {val_results.box.map50:.4f}")
print(f"  mAP@0.5:0.95: {val_results.box.map:.4f}")
print(f"  Precision:     {val_results.box.mp:.4f}")
print(f"  Recall:        {val_results.box.mr:.4f}")
print(f"{'='*60}")

# Per-class AP
print(f"\n  {'Class':<15} {'AP@0.5':>8}")
print(f"  {'-'*25}")
for i, cls in enumerate(VOC_CLASSES):
    if i < len(val_results.box.ap50):
        print(f"  {cls:<15} {val_results.box.ap50[i]:.4f}")

## 4.3 Inference Speed va Save Results

In [ ]:
# Do inference speed
val_images = glob.glob(os.path.join(DATA_ROOT, "VOC_YOLO/images/val/*.jpg"))[:100]

times = []
for img_path in tqdm(val_images, desc="Measuring speed"):
    start = time.time()
    _ = best_model.predict(img_path, imgsz=640, verbose=False)
    times.append(time.time() - start)

times = times[10:]  # Bo warmup
avg_time = np.mean(times)
fps = 1.0 / avg_time

print(f"\nYOLOv8m Inference Speed:")
print(f"  Average time: {avg_time * 1000:.1f} ms/image")
print(f"  FPS: {fps:.1f}")

# Save ket qua
yolov8_results = {
    "mAP_50": float(val_results.box.map50),
    "mAP_50_95": float(val_results.box.map),
    "precision": float(val_results.box.mp),
    "recall": float(val_results.box.mr),
    "fps": float(fps),
    "avg_inference_ms": float(avg_time * 1000),
    "total_params": sum(p.numel() for p in best_model.model.parameters()),
    "per_class": {
        cls: {"ap": float(val_results.box.ap50[i]) if i < len(val_results.box.ap50) else 0}
        for i, cls in enumerate(VOC_CLASSES)
    },
}

with open(os.path.join(RESULTS_DIR, "eval_results.json"), "w") as f:
    json.dump(yolov8_results, f, indent=2)
print("Results saved.")

## 4.4 Visualization

In [ ]:
# Detection samples
np.random.seed(123)
sample_images = np.random.choice(val_images, 6, replace=False)

fig, axes = plt.subplots(2, 3, figsize=(20, 14))
for ax, img_path in zip(axes.flat, sample_images):
    result = best_model.predict(img_path, imgsz=640, conf=0.5, verbose=False)[0]
    img = Image.open(img_path)
    ax.imshow(np.array(img))

    if result.boxes is not None:
        for box, cls, conf in zip(result.boxes.xyxy.cpu(), result.boxes.cls.cpu(), result.boxes.conf.cpu()):
            x1, y1, x2, y2 = box.numpy()
            cls_name = VOC_CLASSES[int(cls)]
            rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2,
                                      edgecolor="lime", facecolor="none")
            ax.add_patch(rect)
            ax.text(x1, y1-3, f"{cls_name}:{conf:.2f}", fontsize=7, color="white",
                    bbox=dict(boxstyle="round,pad=0.2", facecolor="green", alpha=0.8))

    n_det = len(result.boxes) if result.boxes is not None else 0
    ax.set_title(f"{n_det} detections", fontsize=10)
    ax.axis("off")

plt.suptitle("YOLOv8m Detection Results (conf >= 0.5)", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "detection_samples.png"), dpi=150, bbox_inches="tight")
plt.show()

## 4.5 Phan tich YOLOv8

### Cai tien so voi YOLOv3:

1. **Anchor-free**: Khong can predefined anchors -> giam hyperparameters, tang kha nang generalize
2. **Decoupled Head**: Tach classification va regression thanh 2 nhanh rieng -> hoc tot hon vi 2 tasks co nature khac nhau
3. **C2f module**: Cross Stage Partial v2 voi flow connections -> gradient flow tot hon, hoc nhanh hon
4. **CIoU + DFL loss**: 
   - CIoU: Tinh ca overlap, distance, va aspect ratio -> bbox chinh xac hon
   - Distribution Focal Loss (DFL): Predict distribution cua bbox boundaries thay vi single value
5. **Task-Aligned Assignment (TAL)**: Gan label dua tren ca classification score va IoU -> alignment tot hon giua cls va reg
6. **Close mosaic**: Tat mosaic augmentation o cuoi training -> model hoc real distribution tot hon

### Diem manh:
- **State-of-the-art accuracy** tren nhieu benchmark
- **Nhanh**: Anchor-free + efficient architecture
- **Linh hoat**: 5 variants (n, s, m, l, x) cho nhieu use cases
- **API de dung**: Ultralytics cung cap API rat clean

### Diem yeu:
- **Black box hon**: Nhieu techniques chong len nhau, kho biet technique nao dong gop nhieu nhat
- **Training cost**: Can nhieu data va compute hon de tan dung het kha nang
- **Model size**: Lon hon YOLOv3 (cung variant)

### Key insight:
YOLOv8 dai dien cho xu huong **anchor-free + decoupled head** dang thong tri object detection. Viec loai bo anchors khong chi giam do phuc tap ma con giup model generalize tot hon sang datasets khac nhau.